## Predicting price with size

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import wqet_grader
from IPython.display import VimeoVideo
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.utils.validation import check_is_fitted

warnings.simplefilter(action="ignore", category=FutureWarning)

#### Write a function named wrangle that takes a file path as an argument and returns a DataFrame.

In [ ]:
# Add to your wrangle function so that the DataFrame it returns only includes apartments in Buenos Aires ("Capital Federal") 
# that cost less than $400,000 USD. Then recreate df from data/buenos-aires-real-estate-1.csv by re-running the cells above.

def wrangle(filepath):
    # Read CSV file into DataFrame
    df = pd.read_csv(filepath)
    
    # Subset to properties in Capital Federal
    mask_ba = df['place_with_parent_names'].str.contains("Capital Federal")
    # Subset to Apartment
    mask_apt = df["property_type"] == "apartment"
    # Subset to properties where "price_aprox_usd" < 400000
    mask_price = df["price_aprox_usd"] < 400_000
    
    df = df[mask_ba & mask_apt & mask_price]
    
    return df

In [ ]:
# Use your wrangle function to create a DataFrame df from the CSV file data/buenos-aires-real-estate-1.csv.

df = wrangle("data/buenos-aires-real-estate-1.csv")
print("df shape:", df.shape)
df.head()

In [ ]:
# To check your work, df should no have no more than 1,781 observations.

# Check your work
assert (
    len(df) <= 1781
), f"`df` should have no more than 1781 observations, not {len(df)}."

In [ ]:
# Create a histogram of "surface_covered_in_m2". Make sure that the x-axis has the label "Area [sq meters]" 
# and the plot has the title "Distribution of Apartment Sizes".

import matplotlib.pyplot as plt

# Assuming df is your DataFrame
plt.figure(figsize=(10, 6))
plt.hist(df["surface_covered_in_m2"].dropna(), bins=30, edgecolor='black')
plt.xlabel("Area [sq meters]")
plt.ylabel("Frequency")
plt.title("Distribution of Apartment Sizes")
plt.show()


In [ ]:
# Calculate the summary statistics for df using the describe method.
df.describe()["surface_covered_in_m2"]

# The statistics above confirm what we suspected. While most of the apartments in our dataset are smaller that 73 square meters, 
# there are some that are several thousand square meters. 
# The best thing to do is to change our wrangle function and remove them from the dataset.

In [5]:
## Add to your wrangle function so that it removes observations that are outliers in the "surface_covered_in_m2" column. 
# Specifically, all observations should fall between the 0.1 and 0.9 quantiles for "surface_covered_in_m2".

# What's a quantile?
# Calculate the quantiles for a Series in pandas.
# Subset a DataFrame with a mask using pandas.

# When you're done, don't forget to rerun all the cells above. Note how your histogram changes now that there are no outliers. 
# At this point, df should have no more than 1,343 observations**.

In [ ]:
def wrangle(filepath):
    # Read CSV file into DataFrame
    df = pd.read_csv(filepath)
    
    # Subset to properties in Capital Federal
    mask_ba = df['place_with_parent_names'].str.contains("Capital Federal")
    # Subset to Apartment
    mask_apt = df["property_type"] == "apartment"
    # Subset to properties where "price_aprox_usd" < 400000
    mask_price = df["price_aprox_usd"] < 400_000
    # Subset
    df = df[mask_ba & mask_apt & mask_price]
    
    # Remove outliers by surface_covered_in_m2
    low, high = df["surface_covered_in_m2"].quantile([0.1, 0.9])
    mask_area = df["surface_covered_in_m2"].between(low, high)
    df = df[mask_area]

    return df

In [7]:
# Create a scatter plot that shows price ("price_aprox_usd") vs area ("surface_covered_in_m2") in our dataset. 
# Make sure to label your x-axis "Area [sq meters]" and your y-axis "Price [USD]".

plt.figure(figsize=(10, 6))
plt.scatter(df["surface_covered_in_m2"], df["price_aprox_usd"], alpha=0.5)
plt.xlabel("Area [sq meters]")
plt.ylabel("Price [USD]")
plt.title("Price vs. Area")
plt.grid(True)
plt.show()

# This plot suggests that there's a moderate positive correlation between apartment price and size. This means that if thing we want to predict is price, 
# size will be a good feature to include.

## SPLIT

#### A key part in any model-building project is separating your target (the thing you want to predict) from your features (the information your model will use to make its predictions). 
#### Since this is our first model, we'll use just one feature: apartment size.

In [ ]:
## Create the feature matrix named X_train, which you'll use to train your model. It should contain one feature only: ["surface_covered_in_m2"]. 
## Remember that your feature matrix should always be two-dimensional.

features = ["surface_covered_in_m2"]
X_train = df[features]

target = "price_aprox_usd"
y_train = df[target]

# Build Model

### Baseline

In [15]:
# The first step in building a model is baselining. To do this, ask yourself how you will know if the model you build is performing well?
# " One way to think about this is to see how a "dumb" model would perform on the same data. Some people also call this a naïve or baseline model, 
# but it's always a model makes only one prediction — in this case, it predicts the same price regardless of an apartment's size. 
# So let's start by figuring out what our baseline model's prediction should be.

In [ ]:
# Calculate the mean of your target vector y_train and assign it to the variable y_mean

y_mean = y_train.mean()
y_mean

In [ ]:
# Create a list named y_pred_baseline that contains the value of y_mean repeated so that it's the same length at y.

y_pred_baseline = [y_mean] * len(y_train)

In [ ]:
## Add a line to the plot below that shows the relationship between the observations X_train and our dumb model's predictions y_pred_baseline. 
## Be sure that the line color is orange, and that it has the label "Baseline Model".

In [ ]:
plt.plot(X_train["surface_covered_in_m2"], y_pred_baseline, color="orange", label="Baseline Model")
plt.scatter(X_train, y_train)
plt.xlabel("Area [sq meters]")
plt.ylabel("Price [USD]")
plt.title("Buenos Aires: Price vs. Area")
plt.legend();

In [20]:
## Calculate the baseline mean absolute error for your predictions in y_pred_baseline as compared to the true targets in y.

mae_baseline = mean_absolute_error(y_train, y_pred_baseline)

print("Mean apt price", round(y_mean, 2))
print("Baseline MAE:", round(mae_baseline, 2))

# Iterate
#### The next step in building a model is iterating. This involves building a model, training it, evaluating it, 
#### and then repeating the process until you're happy with the model's performance. Even though the model we're building is linear, 
#### the iteration process rarely follows a straight line. Be prepared for trying new things, hitting dead-ends, 
#### and waiting around while your computer does long computations to train your model. ☕️ Let's get started!

#### The first thing we need to do is create our model — in this case, one that uses linear regression


In [24]:
#  Instantiate a LinearRegression model named model.
model = LinearRegression()

In [ ]:
#  Fit your model to the data, X_train and y_train.
model.fit(X_train, y_train)

# Evaluate
### The final step is to evaluate our model. In order to do that, we'll start by seeing how well it performs when making predictions for data that 
### it saw during training. So let's have it predict the price for the houses in our training set.

In [ ]:
y_pred_training = model.predict(X_train)
y_pred_training[:5]

In [ ]:
# Now that we have predictions, we'll use them to assess our model's performance with the training data. We'll use the same metric 
# we used to evaluate our baseline model: mean absolute error.

mae_training = mean_absolute_error(y_train, y_pred_training)
print("Training MAE:", round(mae_training, 2))

In [ ]:
# Extract the intercept from your model, and assign it to the variable intercept.

intercept = round(model.intercept_,2)
print("Model Intercept:", intercept)
assert any([isinstance(intercept, int), isinstance(intercept, float)])

In [ ]:
# Extract the coefficient associated "surface_covered_in_m2" in your model, and assign it to the variable coefficient.

coefficient = round(model.coef_[0],2)
print('Model coefficient for "surface_covered_in_m2":', coefficient)
assert any([isinstance(coefficient, int), isinstance(coefficient, float)])

In [ ]:
# Complete the code below and run the cell to print the equation that your model has determined for predicting apartment price based on size.

# What's an f-string?

print(f"apt_price = {intercept} + {coefficient} * surface_covered)")

In [ ]:
## Add a line to the plot below that shows the relationship between the observations in X_train and your model's predictions y_pred_training. 
## Be sure that the line color is red, and that it has the label "Linear Mode

plt.plot(X_train.values, model.predict(X_train), color="magenta", label="Linear Model")
plt.scatter(X_train, y_train)
plt.xlabel("surface covered [sq meters]")
plt.ylabel("price [usd]")
plt.legend();

# LESSON TWO

In [ ]:
# Use your wrangle function to create a DataFrame frame1 from the CSV file data/buenos-aires-real-estate-1.csv.

frame1 = wrangle("data/buenos-aires-real-estate-1.csv")
print(frame1.info())
frame1.head()

In [ ]:
## Add to the wrangle function below so that, in the DataFrame it returns, the "lat-lon" column is replaced by separate "lat" and "lon" columns. 
# Don't forget to also drop the "lat-lon" column. Be sure to rerun all the cells above before you continue.

def wrangle(filepath):
    # Read CSV file
    df = pd.read_csv(filepath)

    # Subset data: Apartments in "Capital Federal", less than 400,000
    mask_ba = df["place_with_parent_names"].str.contains("Capital Federal")
    mask_apt = df["property_type"] == "apartment"
    mask_price = df["price_aprox_usd"] < 400_000
    df = df[mask_ba & mask_apt & mask_price]

    # Subset data: Remove outliers for "surface_covered_in_m2"
    low, high = df["surface_covered_in_m2"].quantile([0.1, 0.9])
    mask_area = df["surface_covered_in_m2"].between(low, high)
    df = df[mask_area]

    ## split "lat-lon" columns
    df[['lat', 'lon']] = df["lat-lon"].str.split(",", expand=True).astype(float)
    df.drop(columns="lat-lon",inplace=True)

    return df

In [ ]:
## Use pd.concat to concatenate frame1 and frame2 into a new DataFrame df. Make sure you set the ignore_index argument to True

df = pd.concat([frame1, frame2], ignore_index=True)
print(df.info())
df.head()

# Explore
### In the last lesson, we built a simple linear model that predicted apartment price based on one feature, "surface_covered_in_m2". 
### In this lesson, we're building a multiple linear regression model that predicts price based on two features, "lon" and "lat". 
### This means that our data visualizations now have to communicate three pieces of information: Longitude, latitude, and price. 
### How can we represent these three attributes on a two-dimensional screen?

### One option is to incorporate color into our scatter plot. For example, in the Mapbox scatter plot below, the location of each point represents 
### latitude and longitude, and color represents price.

In [ ]:
# Complete the code below to create a Mapbox scatter plot that shows the location of the apartments in df

fig = px.scatter_mapbox(
    df,  # Our DataFrame
    lat="lat",
    lon="lon",
    width=600,  # Width of map
    height=600,  # Height of map
    color="price_aprox_usd",
    hover_data=["price_aprox_usd"],  # Display price when hovering mouse over house
)

fig.update_layout(mapbox_style="open-street-map")

fig.show()

In [ ]:
# Complete the code below to create a 3D scatter plot, with "lon" on the x-axis, "lat" on the y-axis, and "price_aprox_usd" on the z-axis.

# Create 3D scatter plot
fig = px.scatter_3d(
    df,
    x="lon",
    y="lat",
    z="price_aprox_usd",
    labels={"lon": "longitude", "lat": "latitude", "price_aprox_usd": "price"},
    width=600,
    height=500,
)

# Refine formatting
fig.update_traces(
    marker={"size": 4, "line": {"width": 2, "color": "DarkSlateGrey"}},
    selector={"mode": "markers"},
)

# Display figure
fig.show()

# Split
##### Even though we're building a different model, the steps we follow will be the same. Let's separate our features (latitude and longitude) 
##### from our target (price).

In [ ]:
features = ["lon", "lat"]
X_train = df[features]

In [ ]:
imputer = SimpleImputer(strategy="mean")

In [ ]:
# Fit your transformer imputer to the feature matrix X.
imputer.fit(X_train)

In [ ]:
## Use your imputer to transform the feature matrix X_train. Assign the transformed data to the variable XT_train.

XT_train = imputer.transform(X_train)
pd.DataFrame(XT_train, columns=X_train.columns).info()

In [ ]:
# Create a pipeline named model that contains a SimpleImputer transformer followed by a LinearRegression predictor.

model = make_pipeline(
    SimpleImputer(strategy="mean"),
    LinearRegression()
)

In [ ]:
# With our pipeline assembled, we use the fit method, which will train the transformer, transform the data, then pass the transformed data to the predictor for training, all in one step. 
# Much easier!

# Fit your model to the data, X_train and y_train.
model.fit(X_train, y_train)

In [ ]:
# Using your model's predict method, create a list of predictions for the observations in your feature matrix X_train. 
# Name this list y_pred_training.

y_pred_training = model.predict(X_train)

In [ ]:
# Calculate the training mean absolute error for your predictions in y_pred_training as compared to the true targets in y_train

mae_training = mean_absolute_error(y_train, y_pred_training)
print("Training MAE:", round(mae_training, 2))

In [ ]:
# Import test data
X_test = pd.read_csv("data/buenos-aires-test-features.csv")[features]
y_pred_test = pd.Series(model.predict(X_test))
y_pred_test.head()

In [ ]:
# Extract the intercept and coefficients for your model.

intercept = model.named_steps['linearregression'].intercept_
coefficients = model.named_steps['linearregression'].coef_

print("Intercept:", intercept)
print("Coefficients:", coefficients)

In [ ]:
# f-string

print(
    f"price = {intercept} + ({coefficients[0]} * longitude) + ({coefficients[1]} * latitude)"
)


In [ ]:
# Complete the code below to create a 3D scatter plot, with "lon" on the x-axis, "lat" on the y-axis, and "price_aprox_usd" on the z-axis.

fig = px.scatter_3d(
    df,
    x="lon",
    y="lat",
    z="price_aprox_usd",
    labels={"lon": "longitude", "lat": "latitude", "price_aprox_usd": "price"},
    width=600,
    height=500,
)

# Create x and y coordinates for model representation
x_plane = np.linspace(df["lon"].min(), df["lon"].max(), 10)
y_plane = np.linspace(df["lat"].min(), df["lat"].max(), 10)
xx, yy = np.meshgrid(x_plane, y_plane)

# Use model to predict z coordinates
z_plane = model.predict(pd.DataFrame({"lon": xx.ravel(), "lat": yy.ravel()}))
zz = z_plane.reshape(xx.shape)

# Add plane to figure
fig.add_trace(go.Surface(x=xx, y=yy, z=zz, opacity=0.5))

# Refine formatting
fig.update_traces(
    marker={"size": 4, "line": {"width": 2, "color": "DarkSlateGrey"}},
    selector={"mode": "markers"},
)

# Display figure
fig.show()


# LESSON 3
predicting price of house based on neighbourhood

In [ ]:
import warnings
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wqet_grader
from category_encoders import OneHotEncoder
from IPython.display import VimeoVideo
from sklearn.linear_model import LinearRegression, Ridge  # noqa F401
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.utils.validation import check_is_fitted